# Examining the effect of cloud masking algorithms on remote sensing data accuracy and processing to improve water quality in Sandusky Bay, Ohio

Cloud removal is crucial to the improvement of satellite data quality, which can be used to detect spectral signatures of water bodies for the purpose of accurately predicting harmful algal bloom (HAB) events. Specifically, this project seeks to compare the cloud masks and resulting cloud-free composites of popular cloud-masking algorithms against labeled reference data used as ground truth. Each algorithm is implemented using the Harmonized Sentinel-2 Level-2A surface reflectance collection and tested using a total of 15 scenes with varying levels of cloudiness over the Sandusky Bay region of Ohio. Each algorithm masks both opaque and cirrus cloud types, yet only one of them explicitly detects cloud shadows (Cloud Score+). The aim is to determine which algorithm produces the best results for this region of interest and potentially other freshwater environments at risk of environmental harm. In a freshwater environment, cloud shadows and thin clouds are critical, especially as small water bodies are highly sensitive to misclassification. Algorithm selection was based on alignment with objectives, feasibility of implementation, and the results of research papers.

## Initialization

### Imports and installations

In [ ]:
import os
import sys
import ee
import ee.batch
import time
import glob
from pathlib import Path

import rasterio  # Read and write geospatial raster data
import numpy as np
import requests
from dotenv import load_dotenv
import folium  # Visualization (maps)

# ----------------------------------
# For use in image creation for IRIS.
# ----------------------------------
import json
import imagecodecs
import tifffile  # For writing NumPy array to GeoTIFF file.
from tifffile import imread, imwrite

def _looks_like_repo_root(path: Path) -> bool:
    return (path / 'src' / 'utils' / 'config.py').exists() and (path / 'config' / 'config.yaml').exists()

def find_repo_root() -> Path:
    """Find project root for local runs so `src` imports resolve."""
    candidates = []

    # Optional explicit override (recommended).
    env_root = os.getenv('RESEARCH_ROOT')
    if env_root:
        candidates.append(Path(env_root).expanduser())

    # Start from current working directory and walk upward.
    cwd = Path.cwd()
    candidates.extend([cwd, *cwd.parents])

    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)

        if _looks_like_repo_root(resolved):
            return resolved

    raise FileNotFoundError(
        'Could not find repo root containing src/utils/config.py and config/config.yaml. '
        'Set RESEARCH_ROOT in your local environment, e.g. /Users/gjakli/research.'
    )

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Make relative file paths (like config/config.yaml) resolve from repo root.
os.chdir(REPO_ROOT)
print(f'Using repo root: {REPO_ROOT}')

# Load environment variables from repo .env if present.
load_dotenv(REPO_ROOT / '.env')

from src.utils.config import load_config
config = load_config()

# Must authenticate your EE account before use of the package.
project_id = os.getenv('PROJECT_NAME')
ee.Authenticate()
ee.Initialize(project=project_id)

Using repo root: /Users/gjakli/research



Successfully saved authorization token.


### Create EE Image collection

Spectral bands are a necessary component of geospatial analysis. Bands B2, B3, and B4 are essential for both RGB visualization and water quality algorithms. B8, the near-infrared band is critical for cloud detection and water/land discrimination. B11 and B12 are the shortwave infrared bands and provide atmospheric information excellent for cloud detection.

In [2]:
# B1 --> Aerosols (60m)
# B2 --> Blue (10m)
# B3 --> Green (10m)
# B4 --> Red (10m)
# B5 --> Red Edge 1 (20m)
# B6 --> Red Edge 2 (20m)
# B7 --> Red Edge 3 (20m)
# B8 --> NIR (10m)
# B8A --> Red Edge 4 (20m)
# B9 --> Water vapor (60m)
# B11 --> SWIR 1 (20m)
# B12 --> SWIR 2 (20m)

# Select most optimal band combination for water quality analysis.
# water_bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# Subset of bands to be resampled
not_10m = ['B1', 'B5', 'B6', 'B7', 'B8A', 'B9', 'B11', 'B12']

# Select all bands
all_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

The time period for this analysis includes selected dates from 2020 to 2025, and span the warmest months of the year, April through October. Since harmful algal blooms (HABs) thrive in warm, sunlit conditions, this time of year makes the most sense. This research work commenced in early 2023 and concluded in August 2024. The goal at the time was to study the last three years of change for the area of interest. Once I discovered the absence of the QA60 band from February 2022 through February 2024 due to changes in the processing pipeline, the selected dates were adjusted accordingly.

The cloud filter parameter has been initialized to 90 for this project, for the purposes of allowing a greater selection of data (the area is often cloudy and this generates a less exclusive selection), and to allow more cloudy scenes for a more thorough comparison.

After visualizing Sandusky Bay in the Copernicus Browser, a polygon was determined as the most suitable Earth Engine geometry for this region, and coordinates were pulled directly from this resource. Here, EPSG:4326 is the coordinate reference system assumed for projection, the standard for latitude and longitude based on WGS84.

In [4]:
# Define area of interest: Sandusky Bay, Ohio.
AOI = ee.Geometry.Polygon(
    [[[-82.666964,41.620245],
      [-83.062633,41.620245],
      [-83.062633,41.340059],
      [-82.666964,41.340059],
      [-82.666964,41.620245]]
    ]
  )

date_ranges = [
    ('2020-04-01', '2020-05-01'),
    ('2020-07-01', '2020-08-01'),
    # ('2020-08-01', '2020-09-01'),
    # ('2020-09-01', '2020-10-01'),
    # ('2021-04-01', '2021-05-01'),
    # ('2021-06-01', '2021-07-01'),
    # ('2021-08-01', '2021-09-01'),
    # ('2021-10-01', '2021-11-01'),
    # ('2024-05-01', '2024-06-01'),
    # ('2024-06-01', '2024-07-01'),
    # ('2024-08-01', '2024-09-01'),
    # ('2025-05-01', '2025-06-01'),
    # ('2025-06-01', '2025-07-01'),
    # ('2025-07-01', '2025-08-01'),
    # ('2025-08-01', '2025-09-01')
]

# Define the center point for plotting and metadata purposes.
# Coordinates for Folium maps are [latitude, longitude] (reverse of GEE).
center = AOI.centroid(10).coordinates().reverse().getInfo()

# Maximum cloud cover percent allowed in image collection.
CLOUD_FILTER = config['thresholds']['cloud_filter']

### Define reusable functions

#### Visualization settings and implementation for cloud mask and composite images

In [ ]:
# Define a method for displaying Earth Engine image tiles to a folium map.

def add_ee_layer(self, ee_image_object, vis_params, name, show=True, opacity=1, min_zoom=0):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        name=name,
        show=show,
        opacity=opacity,
        min_zoom=min_zoom,
        overlay=True,
        control=True
        ).add_to(self)

# Add the Earth Engine layer method to folium.
folium.Map.add_ee_layer = add_ee_layer

In [ ]:
# Visualize cloud mask components layered over the area of interest.

def display_cloud_mask(col, mask):
    # Mosaic the image collection.
    img = col.mosaic()

    # Subset layers and prepare them for display.
    cloudmask = img.select(mask).selfMask()

    # Create a folium map object.
    m = folium.Map(location=center, zoom_start=12)

    # Add layers to the folium map.
    m.add_ee_layer(img, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2500, 
                'gamma': 1.1},'S2 image', True, 1, 9)
    m.add_ee_layer(cloudmask, {'palette': 'orange'},'cloudmask', True, 0.5, 9)

    # Add a layer control panel to the map.
    m.add_child(folium.LayerControl())

    # Display the map.
    display(m)

In [8]:
# Use with export function to visualize the cloud mask in black and white.

def cloud_mask_vis(image):
  # Define the range of pixel values that will be mapped to 0-255.
  vis = image.visualize(**{
    'palette': ['black', 'white'],
    "min": 0,     # Pixel values at or below this will be displayed as black.
    "max": 0.4    # Pixel values at or above this will be displayed as white.
  })
  return vis

#### Export EE image to Google Drive

In [4]:
# Export to a folder in Google Drive. If raw mask is used as input for
# the image parameter, the output will be binary (values strictly 0 or 1). Else,
# if the cloud_mask_vis function is used with the mask, the output will contain values from 0 to 255.

def export_image(mask, description, folder='Results', region=AOI, scale=10, crs='EPSG:4326', maxPixels=1e9):
    # Export cloud mask to Google Drive.
    task = ee.batch.Export.image.toDrive(
        image=mask,
        description=description,    # Filename.
        # The Google Drive Folder that the export will reside in. Note: 
        # (a) if the folder name exists at any level, the output is written to it, 
        # (b) if duplicate folder names exist, output is written to the most recently modified folder, 
        # (c) if the folder name does not exist, a new folder will be created at the root, and 
        # (d) folder names with separators (e.g. 'path/to/file') are interpreted as literal strings, 
        # not system paths. Defaults to Drive root.
        folder=folder,      
        region=region,
        scale=scale,    # Resolution in meters per pixel. Defaults to 1000.
        crs=crs,      # Default is global GCS; EPSG:32617 is specific to Ohio.
        maxPixels=maxPixels
    )
    task.start()

    print("Export started. Check your Earth Engine Tasks tab.")

    # Monitor the task status
    print(task.status()['id'])
    while task.status()['state'] != 'COMPLETED':
        print(task.status()['state'])
        time.sleep(60)

    print('Done')

#### Preprocess cloud masks for each algorithm by upsampling bands

Reflectance bands and all ancillary bands for all algorithms are resampled (upsampled) to 10m resolution. This is important for avoiding alignment mismatches, i.e., ensuring a fair comparison. Although it requires larger storage and compute, upsampling is preferable to downsampling in this study for two reasons. Spatial detail is crucial for small water features and cloud edges, and 10m resolution allows for preservation of such detail. Also, s2cloudless and Cloud Score+ expect 10 m bands for the visible/NIR inputs, and downsampling may change their behaviour in ways unrelated to the algorithm. Bilinear interpolation is used for continuous reflectance and probability layers to avoid blocky artifacts, while nearest-neighbor is needed for categorical/classification layers to avoid creating mixed fractional classes that break thresholds/logic.

In [5]:
# --- Core preprocessing for reflectance bands ---
def preprocess_reflectance(img):
  """
  Resamples all reflectance bands to 10 m.
  """
  refl_10m = (img.select(not_10m)
                .resample('bilinear')
                .reproject(crs='EPSG:4326', scale=10))
  return img.addBands(refl_10m, overwrite=True)

# --- Hybrid method preprocessing ---
def preprocess_hybrid(img):
  aoi_buffered = AOI.buffer(20)
  # Start with reflectance bands.
  img_out = preprocess_reflectance(img)

  # Resample ancillary bands to 10m.
  cldprb = (img.select('MSK_CLDPRB')
              .resample('bilinear')
              .reproject(crs='EPSG:4326', scale=10)
              .clamp(0, 1))

  classi = (img.select(['MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS'])
              # Nearest-neighbor is default.
              .resample()
              .reproject(crs='EPSG:4326', scale=10))

  # Add the upsampled ancillary bands.
  img_out = img_out.addBands([cldprb, classi], overwrite=True)

  # Clip after all bands are added.
  img_out = img_out.clip(aoi_buffered)

  return img_out

# --- s2cloudless preprocessing ---
def preprocess_s2cloudless(img):
  img_out = preprocess_reflectance(img)

  scl = (img.select('SCL')
            # Nearest-neighbor approach.
            .resample()
            .reproject(crs='EPSG:4326', scale=10))

  img_out = img_out.addBands(scl, overwrite=True)

  return img_out.clip(AOI)

# Resample reflectance bands and clip each image to AOI.
# For use in Cloud Score + algorithm and IRIS input.
def preprocess_reflectance_with_clip(img):
  img_out = preprocess_reflectance(img)
  return img_out.clip(AOI)

## Masking Algorithms

### Probabilistic Cloud Mask and Classification Masks Hybrid

In the COPERNICUS/S2_SR_HARMONIZED dataset, several QA and mask bands are provided. All are derived from the European Space Agency's Sen2Cor processor. QA60 is the standard quality assurance band embedded in all Sentinel-2 L1C products. Essentially, it is a binary classifier and bitmask band. The band has been absent since 2022-01-25 due to changes in the processing pipeline. Introduced to address this issue, the Sentinel-2 Harmonized data collection includes QA60 bands generated from the MSK_CLASSI cloud classification bands. This is the simplest algorithm to implement and is included in nearly every cloud masking algorithm comparison study. The inability to fine-tune this method yields a high omission error, an issue with compounding effects in high-level analyses. For this reason, I decided to test a hybrid approach combining the benefits of both the cloud probability band (MSK_CLDPRB) and the classification bands (MSK_CLASSI), both of which are more accurate than QA60. The idea here is to reduce false positives and missed clouds, offering an approach that is more in line with the other two algorithms being compared.

#### Create and export the cloud mask

In [ ]:
# ------------------------------------------------------------
# Hybrid Approach (cloud probability and classification bands)
# ------------------------------------------------------------
def add_hybrid_cloud_mask(img):
  cldprb = img.select('MSK_CLDPRB')
  opaque = img.select('MSK_CLASSI_OPAQUE')
  cirrus = img.select('MSK_CLASSI_CIRRUS')

  hybrid_threshold = config['thresholds']['hybrid']
  # Threshold the probability band and combine with classification bands (1=clouds).
  qa60_cloud_mask = cldprb.gt(hybrid_threshold).Or(opaque.eq(1)).Or(cirrus.eq(1))
  cloud_band = ee.Image(qa60_cloud_mask).rename('hybrid_mask')
  return img.addBands(cloud_band)

In [ ]:
for start_date, end_date in date_ranges:
    # Define and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
    hybrid_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands + ['MSK_CLDPRB', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS'])
    )

    # Resample bands to 10m resolution for all images in the image collection.
    hybrid_collection = hybrid_col.map(preprocess_hybrid)
    #print('Hybrid bands:', hybrid_collection.first().bandNames().getInfo())

    # Apply the hybrid cloud mask to every image in the upsampled collection.
    upsampled_coll_with_mask = hybrid_collection.map(add_hybrid_cloud_mask)

    # Choose a single image using mosaic compositing.
    hybrid_mask = upsampled_coll_with_mask.select('hybrid_mask').mosaic()

    # Display cloud mask as a layer in a Folium map.
    display_cloud_mask(upsampled_coll_with_mask, 'hybrid_mask')

    # Export cloud mask to Google Drive.
    export_image(hybrid_mask, start_date.replace('-', '')[:-2], 'hybrid')


### S2Cloudless
Developed by a company called Sinergise, s2cloudless is available on Sentinel Hub and in Google Earth Engine. A supervised machine learning method is used, specifically a LightGBM decision tree model, along with the following spectral bands: B1, B2, B4, B5, B8, B8A, B9, B10, B11, B12 (Sentinel-2 Level-1C TOA surface reflectance values). Trained on a global distribution of 13 million scenes, including cloud masks from MAJA, s2cloudless uses a 10m spatial resolution, 160m buffer, and is mono-temporal. The output is a cloud probability map. In cloud probability, higher values are more likely to be clouds or highly reflective surfaces.

The algorithm is implemented as follows. Start by building a collection that combines the harmonized Sentinel-2 surface reflectance and Sentinel-2 cloud probability collections as one. They both need to have similar filters with the same bounds and date. They are joined on the “system:index” property. The result is essentially a copy of the surface reflectance collection with a new property with a value corresponding to the s2cloudless image. The next step is to define the cloud mask component function, for which the s2cloudless probability layer and derived cloud mask are added as bands to an S2 surface reflectance image input. The mask is created by thresholding the probability band.

s2cloudless was selected for comparison due to its reputation as one of the best cloud detection algorithms, low computational requirements, accessability and strong documentation.


In [ ]:
# Initialize parameters and join filtered collections.

CLD_PRB_THRESH = config['thresholds']['s2cloudless']['cloud_probability']   # Cloud probability (%); values greater than are considered cloud (int)
NIR_DRK_THRESH = config['thresholds']['s2cloudless']['nir_darkness']   # Near-infrared reflectance; values less than are considered potential cloud shadow (float)
CLD_PRJ_DIST = config['thresholds']['s2cloudless']['cloud_projection_distance']      # Maximum distance (km) to search for cloud shadows from cloud edges (float)
BUFFER = config['thresholds']['s2cloudless']['buffer']     # Distance (m) to dilate the edge of cloud-identified objects.

#### Add the s2cloudless probability layer and derived cloud mask as bands to an S2 SR image input.

In [ ]:
def add_cloud_bands(img):
    # Get s2cloudless image, subset the probability band.
    cld_prb = ee.Image(img.get('s2cloudless')).select('probability')

    # Condition s2cloudless by the probability threshold value (cloud mask).
    is_cloud = cld_prb.gt(CLD_PRB_THRESH).rename('clouds')

    # Add the cloud probability layer and cloud mask as image bands.
    return img.addBands(ee.Image([cld_prb, is_cloud]))

In [ ]:
def add_shadow_bands(img):
    # Identify dark NIR pixels that are not water (potential cloud shadow pixels).
    SR_BAND_SCALE = 1e4
    dark_pixels = (img.select('B8').lt(NIR_DRK_THRESH*SR_BAND_SCALE)
                    .multiply(img.select('SCL').neq(6)).rename('dark_pixels'))

    # Determine the direction to project cloud shadow from clouds (assumes UTM projection).
    shadow_azimuth = ee.Number(90).subtract(ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')))

    # Project shadows from clouds for the distance specified by the CLD_PRJ_DIST input.
    cld_proj = (img.select('clouds').directionalDistanceTransform(shadow_azimuth, CLD_PRJ_DIST*10)
        .reproject(**{'crs': img.select(0).projection(), 'scale': 100})
        .select('distance')
        .mask()
        .rename('cloud_transform'))

    # Identify the intersection of dark pixels with cloud shadow projection.
    shadows = cld_proj.multiply(dark_pixels).rename('shadows')

    # Add dark pixels, cloud projection, and identified shadows as image bands.
    return img.addBands(ee.Image([dark_pixels, cld_proj, shadows]))

In [ ]:
def cld_mask(img):
    # Add cloud component bands.
    img_cloud = add_cloud_bands(img)

    # Add cloud shadow component bands.
    img_cloud_shadow = add_shadow_bands(img_cloud)

    # Subset cloud and shadow image bands and perform pixel-wise addition. Set their value to 1.
    is_cld_shdw = img_cloud_shadow.select('clouds').add(img_cloud_shadow.select('shadows')).gt(0)

    # Remove small cloud-shadow patches and dilate remaining pixels by BUFFER input.
    # 20 m scale is for speed, and assumes clouds don't require 10 m precision.
    is_cld_shdw = (is_cld_shdw.focalMin(2).focalMax(BUFFER*2/20)
        .reproject(**{'crs': img.select([0]).projection(), 'scale': 20})
        .rename('cloudmask'))

    # Add the final cloud-shadow mask to the image.
    return img_cloud_shadow.addBands(is_cld_shdw)

#### Build the collection. Visualize and export the cloud mask.

In [ ]:
for start_date, end_date in date_ranges:
    # Resample bands to 10m resolution for all images in the image collection.
    s2_sr_col_plus_scl = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands + ['SCL'])
    )

    # Resample reflectance and ancillary bands to 10m resolution for all images in collection.
    upsampled_s2_collection = s2_sr_col_plus_scl.map(preprocess_s2cloudless)
    print('s2cloudless bands:', upsampled_s2_collection.first().bandNames().getInfo())

    # Import and filter s2cloudless.
    s2cloudless_col = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
        .filterBounds(AOI)
        .filterDate(start_date, end_date))

    # Join probability and SR collections on matching property value and name it.
    s2cloudless = ee.ImageCollection(ee.Join.saveFirst('s2cloudless').apply(**{
        'primary': upsampled_s2_collection,
        'secondary': s2cloudless_col,
        'condition': ee.Filter.equals(**{
            'leftField': 'system:index',
            'rightField': 'system:index'
        })
    }))

    # --- Display image and mask component layers ---
    # Apply cloud mask to every image in the collection.
    s2cloudless_final_mask = s2cloudless.map(cld_mask)

    display_cloud_mask(s2cloudless_final_mask, 'cloudmask')

    # --- Export raster image file of cloud mask ---
    # Loading final cloud mask (cloud and shadows).
    s2cloudless_img = s2cloudless_final_mask.mosaic()
    s2cloudless_mask = s2cloudless_img.select('cloudmask')

    # Export the cloud mask to Google Drive.
    export_image(s2cloudless_mask, start_date.replace('-', '')[:-2], 's2cloudless')


### Cloud Score +

Developed by a small group of researchers and geospatial data scientists, Cloud Score + uses a weakly supervised deep learning approach for individual pixel QA. It is generated from the Sentinel-2 L1C, and works equally well on L2A. “Weakly supervised” means that a very small set of manually annotated images with assigned “meaning” to scores, known as an atmospheric similarity index metric (or measure), was augmented by synthetic ones. A bootstrapping procedure leveraging millions of automatically-mined training samples via short video clips were informed by this measure to assign the per-pixel QA (usability) scores. The algorithm was trained with clear references, mean, standard deviation, image, terrain, and metadata, and the ASIM scores are used to create masks by thresholding. Thresholding allows for finetuning and improved results.

The harmonized Sentinel-2 image collection is linked with Cloud Score + using an Earth Engine method called linkCollection(), a single line join to join bands to ee.Image or ee.ImageCollection. Cloud score is a 2-band image in the collection. The two QA bands are cs and cs_cdf. cs can be thought of as a more instantaneous atmospheric similarity score, while cs_cdf captures an expectation of the estimated score through time. The cs_cdf band is selected for this cloud mask experiment because it is less sensitive to thin haze around clouds, offering higher precision, and is therefore more comparable to the other two algorithms in terms of performance. Essentially, this band gives a likelihood of cloud and a tighter, neater boundary.

#### Build collection and generate cloud mask

In [ ]:
# Threshold for masking
CLEAR_THRESHOLD = config['thresholds']['cloudscoreplus']   # CloudScore+ values above this threshold are masked as clouds.
QA_BAND = 'cs_cdf'

In [ ]:
# For visualization, the cloud mask needs to be applied to every image in the collection.
def apply_csplus_mask(img):
  csplus_mask = img.select(QA_BAND).gte(CLEAR_THRESHOLD).Not().rename('csplus_mask')
  return img.addBands(csplus_mask)

#### Visualize and export cloud mask for selected dates

In [ ]:
for start_date, end_date in date_ranges:

  # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
  s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(AOI)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
      .select(all_bands)
      )

  # Resample bands to 10m resolution for all images in the image collection.
  upsampled_csplus_col = s2_sr_col.map(preprocess_reflectance_with_clip)
  #print('Cloud Score + bands:', upsampled_csplus_col.first().bandNames().getInfo())

  # Build Cloud Score + collection with 10m cs_cdf QA band.
  csplus_col = upsampled_csplus_col.linkCollection(
    ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED'),QA_BAND)

  # Apply the mask to every image in the Cloud Score+ collection
  # and visualize as a layer on a Folium map.
  csplus = csplus_col.map(apply_csplus_mask)
  display_cloud_mask(csplus, 'csplus_mask')

  # Initiate a cloud mask object for export to Google Drive.
  image = csplus_col.mosaic()
  csplus_cloud_mask = image.select(QA_BAND).gte(CLEAR_THRESHOLD)
  csplus_cloud_mask_export = csplus_cloud_mask.Not()

  export_image(csplus_cloud_mask_export, start_date.replace('-', '')[:-2], 'cloudscoreplus')

## Generating inputs for IRIS program

IRIS, an active learning software tool by ESA-PhiLab, stands for Intelligently Reinforced Image Segmentation. Cesar Aybar and Ali Francis developed this tool to "accelerate the creation of machine learning training datasets for Earth Observation", as stated on their GitHub repository. Essentially, the software runs locally as a Flask app with a single configuration file to segment multi-spectral and geospatial imagery. Setup and customization are fairly simple. The segmentation process is semi-automatic, as pixels are labeled manually with the support of AI, i.e. a gradient boosted decision tree. Especially useful for cloud segmentation, the data can be transformed into a binary mask containing "clear" and "cloud" classes, or further separated into "cloud shadows", "thick cloud", and "thin cloud". Its purpose in this project is to generate high-quality labeled reference masks necessary for algorithm comparison. No reference masks had been previously generated for Sandusky Bay. Alternative methods were either not feasible, did not offer the same level of quality, or were simply not as convenient.

Primary input is a multi-page TIFF image file containing raw reflectance values of type 'float32'. Thumbnails and metadata are optional, but were used as data validation for each scene. The masks were output in .tif format, encoded as RGB, and scored using F1, as the goal was to achieve balanced performance. A total of 15 scenes of the same area of interest (Sandusky Bay) were segmented, representing various levels of cloudiness, including a variety of cloud types and shadows.

### Initial GeoTIFF image export

Generate the intial GeoTIFF file of our AOI, to use as input for the final .tif file containing an array of raw reflectance values, a necessary component for the IRIS program to function. We are using GeoTIFF to preserve metadata.

In [ ]:
for start, end in date_ranges:
    # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands))

    # Resample all reflectance bands and clip to AOI. Apply to every image in the collection.
    iris_col = s2_sr_col.map(preprocess_reflectance_with_clip)
    #print('Bands:', iris_col.first().bandNames().getInfo())

    # Create the image by compositing all the images in the collection (i.e.,
    # reduces collection to a single image). This method works best for this region
    # to provide complete coverage, since it lies between two areas captured by the satellite.
    iris_img = iris_col.mosaic()

    # Export RGB GeoTIFF file to Google Drive.
    export_image(iris_img, f'GeoTIFF_{start}', 'Iris')

### Export TIFF containing array of raw reflectance values

Using the previously exported GeoTIFF file, a multi-page
32-bit float grayscale .tif file containing the raw reflectance values is created for use as input in the IRIS software, similar to the one used in the demo.

In [ ]:
# Write a 3-dimensional NumPy array to a multi-page grayscale TIFF file.
# Wait until initial GeoTIFF file export is complete to run this section.

for start, end in date_ranges:
  # Creates a 32-bit float image containing raw reflectance values (not scaled to 0-255).
  input_file = f'{config["paths"]["iris_root"]}/GeoTIFF_{start}.tif'
  output_path = f'{config["paths"]["iris_root"]}/raw_{start}.tif'
  data = imread(input_file)

  print(data.shape)
  print(type(data))

  data = data.astype('float32')

  # Save as a multi-page TIFF
  tifffile.imwrite(output_path, data, photometric='minisblack')

  print(f'File saved to Google Drive: {output_path}')

### Optional PNG thumbnail image export

In [ ]:
for start, end in date_ranges:

    # Import and filter Sentinel-2 A/B Harmonized Surface Reflectance dataset.
    s2_sr_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER))
        .select(all_bands))

    iris_col = s2_sr_col.map(preprocess_reflectance_with_clip)

    iris_img = iris_col.mosaic()

    thumbnail_url = iris_img.getThumbURL({
        'min': 0,                 # Pixel values at or below this will be displayed as black.
        'max': 1500,             # Pixel values at or above this will be displayed as white.
        'region': AOI,
        'format': 'png',
        'gamma': 0.8,     # Gamma >1 lightens the image; <1 darkens it.
        'bands': ['B4','B3','B2'],
        'dimensions': 512
    })

    print(thumbnail_url)

    # Download the thumbnail
    thumbnail_path = f'{config["paths"]["iris_root"]}/thumbnail_{start}.png'
    response = requests.get(thumbnail_url)
    with open(thumbnail_path, 'wb') as f:
        f.write(response.content)

### Optional metadata export

In [6]:
for start, end in date_ranges:

    # Create metadata
    metadata = {
        'image_id': start,
        'region': AOI.getInfo(),
        'location': center,
        'bands': all_bands,
        'crs': 'EPSG:4326',
        'start_date': start,
        'end_date': end,
        'cloud_filter': CLOUD_FILTER
    }

    # Export metadata to JSON file
    metadata_path = f'{config["paths"]["iris_root"]}/metadata_{start}.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print("Metadata saved to Google Drive.")

## Processing RGB-encoded GeoTIFF masks from IRIS

After producing the AI-assisted cloud masks in IRIS, further processing is needed before use in the comparison. The IRIS reference masks have no georeferencing, meaning no spatial information is assigned. They are RGB-encoded, with "no cloud" pixels having a value of 255 and "cloud" pixels set as 0. This is the exact opposite of how the masks are labeled for all three algorithms being compared. Additionally, data type is int64. For these reasons, we need to process these GeoTIFF files prior to analysis.

In [6]:
from pathlib import Path

p = Path("/content/gdrive/My Drive/Results/masks/cloudscoreplus/202007.tif")
for i in range(1, len(p.parts)+1):
    s = Path(*p.parts[:i])
    print(f"{s} | exists={s.exists()} | dir={s.is_dir()} | file={s.is_file()}")

/ | exists=True | dir=True | file=False
/content | exists=True | dir=True | file=False
/content/gdrive | exists=True | dir=True | file=False
/content/gdrive/My Drive | exists=True | dir=True | file=False
/content/gdrive/My Drive/Results | exists=True | dir=True | file=False
/content/gdrive/My Drive/Results/masks | exists=True | dir=True | file=False
/content/gdrive/My Drive/Results/masks/cloudscoreplus | exists=True | dir=True | file=False
/content/gdrive/My Drive/Results/masks/cloudscoreplus/202007.tif | exists=True | dir=False | file=True


In [6]:
# Choose any cloud mask file from any of three algorithms to copy metadata.
# This is crucial to ensure that the reference masks are properly georeferenced.
reference_mask_path = '/content/gdrive/My Drive/202004.tif'

with rasterio.open(reference_mask_path) as ref:
    ref_meta = ref.meta.copy()

# Output cloud mask files from the IRIS program need to be uploaded to this directory
# in Google Drive before running the code below to convert them to georeferenced binary
# masks that align with the other two algorithms and can be used for evaluation.
iris_files = glob.glob(f'{config["iris_masks_dir"]}/reference/*.tif')

for iris_input_path in iris_files:
  iris_output_path = iris_input_path.replace(".tif", "_processed.tif")

  with rasterio.open(iris_input_path) as mask:
    # Mask information is in 3rd band (layer).
    iris_data = np.array(mask.read(3))
    # Convert 255 to 0 (no cloud), else 1 (cloud) and convert data type
    # to align with masks produced by the three algorithms.
    iris = np.where(iris_data == 255, 0, 1).astype(rasterio.uint8)

  # Open each mask file and write the newly processed
  # and georeferenced version to the output path.
  with rasterio.open(iris_output_path, "w", **ref_meta) as dest:
    # Write the NumPy array to the first band (band index starts at 1)
    dest.write(iris, 1)
    print("Unique values in saved mask:", np.unique(iris))
    print("CRS:", dest.crs)
    print("Transform:", dest.transform)


  print(f"Saved {iris_input_path} to: {iris_output_path}")

Unique values in saved mask: [0 1]
CRS: EPSG:4326
Transform: | 0.00, 0.00,-83.06|
| 0.00,-0.00, 41.62|
| 0.00, 0.00, 1.00|
Saved /content/gdrive/My Drive/Results/masks/reference/202009.tif to: /content/gdrive/My Drive/Results/masks/reference/202009_processed.tif
Unique values in saved mask: [0 1]
CRS: EPSG:4326
Transform: | 0.00, 0.00,-83.06|
| 0.00,-0.00, 41.62|
| 0.00, 0.00, 1.00|
Saved /content/gdrive/My Drive/Results/masks/reference/202004.tif to: /content/gdrive/My Drive/Results/masks/reference/202004_processed.tif
Unique values in saved mask: [0 1]
CRS: EPSG:4326
Transform: | 0.00, 0.00,-83.06|
| 0.00,-0.00, 41.62|
| 0.00, 0.00, 1.00|
Saved /content/gdrive/My Drive/Results/masks/reference/202007.tif to: /content/gdrive/My Drive/Results/masks/reference/202007_processed.tif
Unique values in saved mask: [0 1]
CRS: EPSG:4326
Transform: | 0.00, 0.00,-83.06|
| 0.00,-0.00, 41.62|
| 0.00, 0.00, 1.00|
Saved /content/gdrive/My Drive/Results/masks/reference/202008.tif to: /content/gdrive/My

In [9]:
# For this section to work, you must first download the processed IRIS mask files
# locally to your device, then manually upload them in the Assets tab of GEE.

# Specify the path to your folder or image collection.
asset_path = f'projects/{project_id}/assets/'

# List assets in the specified path.
asset_list = ee.data.listAssets(asset_path)

# Iterate through the list, print asset details, and export each as a mask file.
for asset in asset_list['assets']:
  iris_mask = ee.Image(asset['id'])

  # Rename the files to avoid overwriting when exporting
  asset_name = asset['id'].split('/')[-1]
  export_name = asset_name.replace('processed', 'mask')

  export_image(cloud_mask_vis(iris_mask), export_name, 'reference')

Export started. Check your Earth Engine Tasks tab.
EOUCL7SPQOLTCGQGPH45YH6F
READY
RUNNING
RUNNING
RUNNING
RUNNING
RUNNING
Done
Export started. Check your Earth Engine Tasks tab.
AJ4PHXEO5JIRFLURFYRB7Y6Q
READY
RUNNING
RUNNING
RUNNING
RUNNING
Done
Export started. Check your Earth Engine Tasks tab.
HPQQFTOCCTEOAPI67RSDBNUB
READY
RUNNING
RUNNING
RUNNING
RUNNING
Done
Export started. Check your Earth Engine Tasks tab.
7ZC37M364WXWPJFPC2PCBTJX
READY
RUNNING
RUNNING
RUNNING
RUNNING
Done


In [5]:
img = ee.Image(f'projects/{project_id}/assets/202007_processed')

# Define the region of interest (the image footprint)
region = img.geometry()

# Count total pixels (non-masked)
total_pixels_dict = img.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=region,
    scale=20,
    maxPixels=1e9
)

# Cloudy pixels = sum of mask values (assuming cloudy = 1, clear = 0)
cloudy_pixels_dict = img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=20,
    maxPixels=1e9
)

# Extract the numerical values from the dictionaries as Earth Engine Numbers
total_pixels = ee.Number(total_pixels_dict.values().get(0))
cloudy_pixels = ee.Number(cloudy_pixels_dict.values().get(0))

# Clear pixels = total - cloudy
clear_pixels = total_pixels.subtract(cloudy_pixels)

# Calculate percentage cloudy using Earth Engine Number operations
percentage_cloudy = (cloudy_pixels.divide(total_pixels)).multiply(100)

# Print results
print("Total pixels:", total_pixels.getInfo())
print("Cloudy pixels:", cloudy_pixels.getInfo())
print("Clear pixels:", clear_pixels.getInfo())
print(f"Percentage cloudy: {percentage_cloudy.getInfo():.2f}%")
print("Data type:", img.bandTypes().getInfo())

Total pixels: 3438883
Cloudy pixels: 3081801
Clear pixels: 357082
Percentage cloudy: 89.62%
Data type: {'b1': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}}
